# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/arjelmilan/flyrank-ai/blob/main/work/notebooks/w03_feature_leakage_check.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Build the feature vector

*Code that actually builds it — engineered features, categorical handling, fills.*

In [1]:
import pandas as pd
import numpy as np

url = "https://raw.githubusercontent.com/arjelmilan/flyrank-ai/main/data/raw/content_refresh_anonymized.csv"

df = pd.read_csv(url)
df["pos_clean"] = np.where(df["avg_position"] == 0, 50.0, df["avg_position"])
df["traffic_drop"] = np.maximum(0, df["clicks_prev_30d"] - df["clicks_last_30d"])

feature_cols = ["impressions_90d", "clicks_prev_30d", "pos_clean", "content_age_days", "ctr"]
X = df[feature_cols].copy()
print(f"Feature matrix shape: {X.shape} across {len(feature_cols)} leak-free features.")


Feature matrix shape: (30000, 5) across 5 leak-free features.


## 2. Feature notes (meaning, missing, categorical, available-when?)

*For each feature: what it means, how missing values are handled, and whether it exists BEFORE the moment you predict.*

In [2]:
feature_notes = {
    "impressions_90d": "90-day search impression volume (Available before prediction window)",
    "clicks_prev_30d": "Prior 30-day baseline click volume (Available before prediction window)",
    "pos_clean": "Clean average search rank position (Imputed missing rank pos=0 to default 50.0)",
    "content_age_days": "Article age in days since publication",
    "ctr": "Click-through rate percentage (clicks / impressions * 100)"
}
for feat, note in feature_notes.items():
    print(f"- {feat}: {note}")


- impressions_90d: 90-day search impression volume (Available before prediction window)
- clicks_prev_30d: Prior 30-day baseline click volume (Available before prediction window)
- pos_clean: Clean average search rank position (Imputed missing rank pos=0 to default 50.0)
- content_age_days: Article age in days since publication
- ctr: Click-through rate percentage (clicks / impressions * 100)


## 3. The leakage hunt

*Attack your own features: label-derived columns, future windows, product flags. Show the test.*

In [3]:

forbidden_leakage_cols = ["trend_direction", "trend_pct", "is_declining_label"]
leakage_in_features = [col for col in forbidden_leakage_cols if col in feature_cols]
assert len(leakage_in_features) == 0, "Temporal data leakage detected!"
print("Leakage Hunt Test: PASSED. Zero target-derived features in feature matrix.")

Leakage Hunt Test: PASSED. Zero target-derived features in feature matrix.


In [6]:
leakage_in_features

[]

## 4. What I excluded and why

*The list of fields you refused to use — with one line of why each.*

In [7]:

exclusions = {
    "trend_direction": "Derived from future target observation window (causes data leakage)",
    "trend_pct": "Summary target percentage drop calculated after the prediction window",
    "is_declining_label": "Binary target classification label provided in ground truth"
}
for col, reason in exclusions.items():
    print(f"- EXCLUDED {col}: {reason}")

- EXCLUDED trend_direction: Derived from future target observation window (causes data leakage)
- EXCLUDED trend_pct: Summary target percentage drop calculated after the prediction window
- EXCLUDED is_declining_label: Binary target classification label provided in ground truth


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.